# Pipeline 5 - Safehouse Capacity and Incident Forecast

Generated: 2026-04-07T11:53:52.838527Z


## 1) Problem Framing

**Business question:** Forecast next-month incident and capacity pressure for safehouse planning.

**Who cares:** nonprofit leadership, program staff, and/or fundraising staff depending on the pipeline domain.

**Why it matters:** this model turns operational, donor, or outreach data into a decision-support signal for a resource-constrained safehouse nonprofit.

**Predictive vs. explanatory goal:** this notebook includes both. The predictive model is evaluated on unseen data and is used for deployment-oriented scoring. The explanatory or relationship model is included to identify which variables appear most connected to the target and to support business interpretation. We do not treat predictive accuracy as causal proof.

**Success metrics:** classification pipelines use accuracy, F1, and ROC AUC where appropriate. Regression/forecasting pipelines use MAE, RMSE, and R-squared. The notebook also compares against a simple baseline so the results can be interpreted honestly.


## Notebook Setup

Shared imports and helper functions are defined once here so the later rubric sections can focus on the pipeline-specific code for this business problem.


In [ ]:
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline

from data_prep import (
    compare_profiles,
    dataset_profile,
    export_predictions_json,
    fill_numeric_median,
    get_project_paths,
    load_df,
    prep,
    quick_eda,
    score_bands,
    time_split,
)
from modeling import (
    CandidateResult,
    bootstrap_prediction_intervals,
    cv_evaluate_candidate,
    cv_results_table,
    eval_regression,
    evaluate_candidate,
    read_json_if_exists,
    select_simplest_within_delta,
    select_simplest_within_delta_cv,
    should_export_by_guardrail,
    tune_model_randomized,
    write_ablation_report_json,
    write_run_report_json,
    ablate_feature_groups_one_by_one_cv,
)

paths = get_project_paths()
print("Data dir:", paths.data_dir)
print("Output dir:", paths.out_dir)


def print_business_takeaway(text):
    print("\nBusiness takeaway:")
    print(text)


## 2) Data Acquisition, Preparation, and Exploration

The pipeline reads the provided CSV files from `data/raw/`, performs joins and feature engineering in code, handles missing values reproducibly, and prints an EDA summary before modeling. The EDA step checks row counts, missingness, target distributions, feature summaries, and key categorical distributions.


In [ ]:
metrics = load_df("safehouse_monthly_metrics")
metrics["month_start"] = pd.to_datetime(metrics["month_start"], errors="coerce")
metrics = metrics.dropna(subset=["month_start"]).sort_values(["safehouse_id","month_start"]).copy()
num_base = ["active_residents","avg_education_progress","avg_health_score","process_recording_count","home_visitation_count","incident_count"]
fill_numeric_median(metrics, num_base)
g = metrics.groupby("safehouse_id")
metrics["incident_lag1"] = g["incident_count"].shift(1)
metrics["incident_lag2"] = g["incident_count"].shift(2)
metrics["active_lag1"] = g["active_residents"].shift(1)
metrics["active_lag2"] = g["active_residents"].shift(2)
metrics["rolling_incident_3m"] = g["incident_count"].transform(lambda s: s.rolling(3, min_periods=1).mean())
metrics["incident_next"] = g["incident_count"].shift(-1)
metrics["active_residents_next"] = g["active_residents"].shift(-1)
df = metrics.dropna(subset=["incident_next","active_residents_next","incident_lag1","active_lag1"]).copy()
features = num_base + ["incident_lag1","incident_lag2","active_lag1","active_lag2","rolling_incident_3m"]
fill_numeric_median(df, features + ["incident_next","active_residents_next"])
train, test = time_split(df, "month_start")
print("Rows:", len(df), "Train:", len(train), "Test:", len(test))
quick_eda(df, "Safehouse forecast modeling table", target_col="incident_next", numeric_cols=features + ["active_residents_next"], categorical_cols=["safehouse_id"])


## 3) Modeling and Feature Selection

The feature set is selected from fields that would be available at the decision point whenever possible. Predictive models use ensembles or tuned tree-based models when they improve out-of-sample performance. Explanatory models use simpler linear or logistic models when interpretability matters.


In [ ]:
# Candidate pool for resource-aware selection.
# We keep models lightweight because this pipeline may run frequently.
incident_candidates = {
    "LinearRegression": LinearRegression(),
    "GradientBoosting": GradientBoostingRegressor(random_state=42),
    "RandomForestSmall": RandomForestRegressor(n_estimators=200, random_state=42, min_samples_leaf=3),
}
active_candidates = {
    "LinearRegression": LinearRegression(),
    "GradientBoosting": GradientBoostingRegressor(random_state=42),
    "RandomForestSmall": RandomForestRegressor(n_estimators=200, random_state=42, min_samples_leaf=3),
}



## 4) Evaluation and Selection

The notebook uses a train/test or time-based holdout split depending on the business problem. Where appropriate, it also uses compact cross-validation, model comparison tables, and lightweight randomized tuning. Metrics are interpreted in business terms rather than treated as abstract statistics.


In [ ]:
# Strict holdout discipline: CV on train, single touch on time-ordered holdout.
X_train, X_holdout = train[features], test[features]

report_path = paths.reports_dir / "safehouse-capacity-forecast_run_report.json"
prev_report = read_json_if_exists(report_path)

current_profile = dataset_profile(train[features], categorical_cols=[], numeric_cols=features)
drift = compare_profiles(current_profile, (prev_report or {}).get("data_profile"))

# Baselines on holdout
print("Incident baseline (lag1):", eval_regression(test["incident_next"], np.maximum(0, test["incident_lag1"].to_numpy())))
print("Capacity baseline (lag1):", eval_regression(test["active_residents_next"], np.maximum(0, test["active_lag1"].to_numpy())))

# 1) CV tournament (incident)
inc_cv_results = [cv_evaluate_candidate(name, est, X_train, train["incident_next"], task="regression") for name, est in incident_candidates.items()]
inc_cv_tbl = cv_results_table(inc_cv_results).sort_values(by=["mae_mean", "fit_s_mean"], ascending=True)
print("Incident CV tournament:")
print(inc_cv_tbl.to_string(index=False))

inc_best = float(inc_cv_tbl["mae_mean"].min())
inc_delta = 0.05 * inc_best
inc_selected_cv = select_simplest_within_delta_cv(inc_cv_results, primary_metric="mae", delta=inc_delta, higher_is_better=False)
inc_base = inc_selected_cv.estimator
inc_base_name = inc_selected_cv.name

# tune incident family
inc_param = {}
if inc_base_name.startswith("RandomForest"):
    inc_param = {"n_estimators": [100, 150, 200, 300], "max_depth": [None, 3, 5, 8], "min_samples_leaf": [1, 3, 5, 8]}
elif inc_base_name.startswith("GradientBoosting"):
    inc_param = {"n_estimators": [100, 150, 250], "learning_rate": [0.03, 0.05, 0.1], "max_depth": [2, 3, 4]}

if inc_param:
    tune_inc = tune_model_randomized(inc_base_name, inc_base, X_train, train["incident_next"], param_distributions=inc_param, task="regression", cv_metric="neg_mean_absolute_error", n_iter=10)
    incident_model = tune_inc.best_estimator
    incident_model_name = f"{inc_base_name}+tuned"
    incident_params = tune_inc.best_params
else:
    incident_model = inc_base
    incident_model_name = inc_base_name
    incident_params = {}

# 2) CV ablation (incident)
inc_ablation = ablate_feature_groups_one_by_one_cv(
    base_name=incident_model_name,
    estimator=incident_model,
    X=X_train,
    y=train["incident_next"],
    task="regression",
    feature_groups=[[c] for c in features],
    primary_metric="mae",
    higher_is_better=False,
)

# 3) CV tournament (capacity)
act_cv_results = [cv_evaluate_candidate(name, est, X_train, train["active_residents_next"], task="regression") for name, est in active_candidates.items()]
act_cv_tbl = cv_results_table(act_cv_results).sort_values(by=["mae_mean", "fit_s_mean"], ascending=True)
print("Capacity CV tournament:")
print(act_cv_tbl.to_string(index=False))

act_best = float(act_cv_tbl["mae_mean"].min())
act_delta = 0.05 * act_best
act_selected_cv = select_simplest_within_delta_cv(act_cv_results, primary_metric="mae", delta=act_delta, higher_is_better=False)
act_base = act_selected_cv.estimator
act_base_name = act_selected_cv.name

act_param = {}
if act_base_name.startswith("RandomForest"):
    act_param = {"n_estimators": [100, 150, 200, 300], "max_depth": [None, 3, 5, 8], "min_samples_leaf": [1, 3, 5, 8]}
elif act_base_name.startswith("GradientBoosting"):
    act_param = {"n_estimators": [100, 150, 250], "learning_rate": [0.03, 0.05, 0.1], "max_depth": [2, 3, 4]}

if act_param:
    tune_act = tune_model_randomized(act_base_name, act_base, X_train, train["active_residents_next"], param_distributions=act_param, task="regression", cv_metric="neg_mean_absolute_error", n_iter=10)
    active_model = tune_act.best_estimator
    active_model_name = f"{act_base_name}+tuned"
    active_params = tune_act.best_params
else:
    active_model = act_base
    active_model_name = act_base_name
    active_params = {}

act_ablation = ablate_feature_groups_one_by_one_cv(
    base_name=active_model_name,
    estimator=active_model,
    X=X_train,
    y=train["active_residents_next"],
    task="regression",
    feature_groups=[[c] for c in features],
    primary_metric="mae",
    higher_is_better=False,
)

# Write ablation reports
write_ablation_report_json(paths.reports_dir / "safehouse-capacity-forecast_incident_ablation_report.json", {"pipeline": "safehouse-capacity-forecast", "target": "incident_next", "table": inc_ablation.to_dict(orient="records")})
write_ablation_report_json(paths.reports_dir / "safehouse-capacity-forecast_capacity_ablation_report.json", {"pipeline": "safehouse-capacity-forecast", "target": "active_residents_next", "table": act_ablation.to_dict(orient="records")})

# Final holdout evaluation
incident_model.fit(X_train, train["incident_next"])
active_model.fit(X_train, train["active_residents_next"])
inc_pred = np.maximum(0, incident_model.predict(X_holdout))
act_pred = np.maximum(0, active_model.predict(X_holdout))
inc_holdout = eval_regression(test["incident_next"], inc_pred)
act_holdout = eval_regression(test["active_residents_next"], act_pred)
inc_pi = bootstrap_prediction_intervals(test["incident_next"], inc_pred, quantiles=(0.10, 0.90))
act_pi = bootstrap_prediction_intervals(test["active_residents_next"], act_pred, quantiles=(0.10, 0.90))

allow_export, guardrail = should_export_by_guardrail(
    previous_report=prev_report,
    current_holdout={"mae": float(inc_holdout["mae"])},
    primary_metric="mae",
    min_delta_allowed=0.05 * float((prev_report or {}).get("incident", {}).get("holdout", {}).get("mae", inc_holdout["mae"])),
    higher_is_better=False,
)

write_run_report_json(
    report_path,
    {
        "pipeline": "safehouse-capacity-forecast",
        "primary_metric": "mae",
        "incident": {
            "selected_family": inc_base_name,
            "tuned_name": incident_model_name,
            "tuned_params": incident_params,
            "cv_table": inc_cv_tbl.to_dict(orient="records"),
            "holdout": inc_holdout,
            "holdout_pi": inc_pi,
        },
        "capacity": {
            "selected_family": act_base_name,
            "tuned_name": active_model_name,
            "tuned_params": active_params,
            "cv_table": act_cv_tbl.to_dict(orient="records"),
            "holdout": act_holdout,
            "holdout_pi": act_pi,
        },
        "guardrail": guardrail,
        "data_profile": current_profile,
        "drift": drift,
    },
)
print("Wrote run report:", report_path)

if not allow_export:
    print("Export blocked by guardrail:", guardrail)



## 5) Causal and Relationship Analysis

The relationship analysis section highlights important features and discusses whether those relationships make sense for the organization. These findings are observational: they can guide hypotheses and strategy, but they are not automatically causal. Any sensitive resident-care decision must remain human-reviewed.


In [ ]:
print("Top incident forecast features (if available):")
if hasattr(incident_model, "feature_importances_"):
    print(
        pd.DataFrame({"feature": features, "importance": incident_model.feature_importances_})
        .sort_values("importance", ascending=False)
        .head(10)
        .to_string(index=False)
    )
else:
    print("(No feature_importances_ on selected incident model)")

print_business_takeaway(
    "Use the safehouse forecast as planning guidance. Use prediction intervals to communicate uncertainty, "
    "and treat outputs as approximate rather than exact counts."
)


## 6) Deployment Notes

The final scoring step exports JSON to `output/ml-predictions/`. These files match the API import contract used by `POST /api/ml/import?replace=true` and can be viewed in the deployed staff portal under `/app/ml` or the ML action center.


In [ ]:
latest = metrics.groupby("safehouse_id").tail(1).copy()
fill_numeric_median(latest, features)

latest["predicted_incidents_next_month"] = np.maximum(0, incident_model.predict(latest[features]))
latest["predicted_active_residents_next_month"] = np.maximum(0, active_model.predict(latest[features]))

# Use holdout-calibrated residual intervals (additive) as a cheap uncertainty estimate.
latest["incidents_p10"] = np.maximum(0, latest["predicted_incidents_next_month"] + inc_pi.get("resid_q_low", 0.0))
latest["incidents_p90"] = np.maximum(0, latest["predicted_incidents_next_month"] + inc_pi.get("resid_q_high", 0.0))
latest["active_p10"] = np.maximum(0, latest["predicted_active_residents_next_month"] + act_pi.get("resid_q_low", 0.0))
latest["active_p90"] = np.maximum(0, latest["predicted_active_residents_next_month"] + act_pi.get("resid_q_high", 0.0))

latest["risk_band"] = score_bands(latest["predicted_incidents_next_month"])

if allow_export:
    export_predictions_json(
        "safehouse_incident_next_month",
        "Safehouse",
        latest[[
            "safehouse_id",
            "predicted_incidents_next_month",
            "incidents_p10",
            "incidents_p90",
            "risk_band",
            "predicted_active_residents_next_month",
            "active_p10",
            "active_p90",
            "active_residents",
            "incident_count",
            "avg_health_score",
            "avg_education_progress",
        ]].assign(incident_model=incident_model_name, capacity_model=active_model_name),
        "safehouse_id",
        "predicted_incidents_next_month",
        "risk_band",
    )
else:
    print("Skipping export due to guardrail.")
